# Defining methods

In [1]:
import numpy as np
import scipy as sc
import scipy.sparse as sp
from petsc4py import PETSc
from slepc4py import SLEPc

def create_scipy_sparse_matrix(data):
    row = data[:, 0].astype(int)
    col = data[:, 1].astype(int)
    val = data[:, 2]
    size = int(max(row.max(), col.max()) + 1)
    return sp.csr_matrix((val, (row, col)), shape=(size, size))

# Function to create PETSc sparse matrix from data
def create_petsc_matrix(data):
    # Extract rows, columns, and values
    rows = data[:, 0].astype(int) - 1  # Convert to zero-based index
    cols = data[:, 1].astype(int) - 1  # Convert to zero-based index
    values = data[:, 2]

    # Create PETSc matrix
    nrows = rows.max() + 1
    ncols = cols.max() + 1
    A = PETSc.Mat().createAIJ([nrows, ncols], nnz=3)
    A.setOption(PETSc.Mat.Option.NEW_NONZERO_ALLOCATION_ERR, False)
    A.setUp()
    for r, c, v in zip(rows, cols, values):
        A[r, c] = v
    A.assemble()
    return A

# Function to convert a SciPy sparse matrix to a PETSc matrix
def scipy_to_petsc(scipy_matrix):
    # coo = scipy_matrix.tocoo()
    # petsc_matrix = PETSc.Mat().createAIJ(size=scipy_matrix.shape, csr=(coo.row.astype(np.int32), coo.col.astype(np.int32), coo.data))
    # petsc_matrix.setValuesCSR(coo.row, coo.col, coo.data)
    petsc_mat = PETSc.Mat().createAIJ(size=scipy_matrix.shape, csr=(scipy_matrix.indptr, scipy_matrix.indices, scipy_matrix.data))
    petsc_mat.assemble()
    return petsc_mat

def compute_invdiag(A):
    diagA = A.getDiagonal()
    diagA_inv = diagA.copy()

    # Compute the lumped row reciprocal of each diagonal entry manually
    for i in range(diagA_inv.size):
        rowsum = 0.0
        for j in range(diagA_inv.size):
            rowsum += A[i, j]
        if rowsum != 0:
            diagA_inv[i] = 1.0 / rowsum
        else:
            print("Row sum is zero for row", i)
            break

    # Create a diagonal matrix D_inv from the inverse diagonal values
    D_inv = PETSc.Mat().createAIJ(A.getSize(), comm=comm)
    D_inv.setUp()
    for i in range(diagA_inv.size):
        D_inv.setValue(i, i, diagA_inv[i])
    D_inv.assemble()
    
    return D_inv

In [2]:
from mpi4py import MPI
import sys

comm = MPI.COMM_WORLD

def par_print(string):
    if comm.rank == 0:
        print(string)
        sys.stdout.flush()

def eigenvalues(A, B, shift=0.0, n_eigs=11, precond = False):
    # Check if need precond
    if precond == True:
        D_inv = compute_invdiag(A+B)
        # D_inv = compute_invdiag(A)
        A = D_inv.matMult(A)
        B = D_inv.matMult(B)


    # Create SLEPc Eigenvalue solver
    eps = SLEPc.EPS().create(comm)
    eps.setOperators(A, B)
    eps.setType(SLEPc.EPS.Type.KRYLOVSCHUR)
    eps.setProblemType(SLEPc.EPS.ProblemType.GHEP)
    eps.setWhichEigenpairs(eps.Which.TARGET_MAGNITUDE)
    eps.setTarget(shift)

    st = eps.getST()
    st.setType(SLEPc.ST.Type.SINVERT)
    st.setShift(shift)

    eps.setDimensions(n_eigs, PETSc.DECIDE, PETSc.DECIDE)
    eps.setFromOptions()
    eps.solve()

    its = eps.getIterationNumber()
    eps_type = eps.getType()
    n_ev, n_cv, mpd = eps.getDimensions()
    tol, max_it = eps.getTolerances()
    tol = 1e-16
    n_conv = eps.getConverged()

    par_print(f"Number of iterations: {its}")
    par_print(f"Solution method: {eps_type}")
    par_print(f"Number of requested eigenvalues: {n_ev}")
    par_print(f"Stopping condition: tol={tol}, maxit={max_it}")
    par_print(f"Number of converged eigenpairs: {n_conv}")

    computed_eigenvalues = []
    for i in range(min(n_conv, n_eigs)):
        lmbda = eps.getEigenvalue(i)
        # computed_eigenvalues.append(np.round(np.real(lmbda), 1))
        computed_eigenvalues.append(np.round(np.real(lmbda), 5))

    eps.destroy()

    # Print results
    # np.set_printoptions(formatter={'float': '{:5.1f}'.format})
    eigenvalues_exact = np.array([2.0, 2.0, 2.0, 3.0, 3.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0])
    computed_eigenvalues = np.sort(computed_eigenvalues)
    
    par_print(f"Exact    = {eigenvalues_exact}")
    par_print(f"Nédélec  = {computed_eigenvalues}")
    return computed_eigenvalues

In [3]:
def preprocess_A_3field(A_data, num_edges, num_faces, num_cells):
    A = create_scipy_sparse_matrix(A_data)

    # Extract submatrices W, M, and B from A
    M = -A[1:num_edges, 1:num_edges]  # edges x edges
    W = A[num_edges:num_edges+num_faces, 1:num_edges]  # faces x edges
    B = A[num_edges + num_faces:num_edges + num_faces + num_cells, num_edges:num_edges+num_faces]  # cells x faces
    # print(M.shape, W.shape, B.shape)

    # Compute W * M^{-1}
    WT = W.transpose()  # edges x faces
    M_invWT = sc.sparse.linalg.spsolve(M, WT)  # edges x faces

    # Compute W * M^{-1} * W^T
    WMWT = W @ M_invWT

    # Transpose B to get B^T
    BT = B.transpose()

    # Determine the sizes for the resulting block matrix
    m, n = WMWT.shape  # Assume WMWT is mxm, m=num_faces
    p, q = B.shape  # Assume B is pxm, p=num_cells

    # Create the resulting block matrix
    result = sp.lil_matrix((m+p, m+p))

    # Insert submatrices into the block matrix
    # Insert W * M^{-1} * W^T into the top-left block
    result[:m, :m] = WMWT

    # Insert B^T into the top-right block
    result[:m, m:m + p] = BT

    # Insert B into the bottom-left block
    result[m:m + p, :m] = B

    # Convert the resulting block matrix to PETSc
    petsc_result = scipy_to_petsc(result.tocsr())

    return petsc_result

def preprocess_B_3field(B_data, num_edges, num_faces, num_cells):
    B = create_petsc_matrix(B_data)

    # Define row and column indices for the submatrices
    row_indices_M = list(range(num_edges, num_edges+num_faces+num_cells))
    col_indices_M = list(range(num_edges, num_edges+num_faces+num_cells))

    # Convert indices to PETSc IS (Index Set)
    row_IS_M = PETSc.IS().createGeneral(row_indices_M)
    col_IS_M = PETSc.IS().createGeneral(col_indices_M)

    # Extract submatrices M from A
    # M = B.createSubMatrix(row_IS_M, col_IS_M)

    # Create the resulting block matrix
    block_size = num_faces + num_cells
    result = PETSc.Mat().createAIJ([block_size, block_size])
    result.setUp()

    # Insert submatrices into the block matrix
    for i in range(block_size):
        for j in range(block_size):
            result[i, j] = B[num_edges+i, num_edges+j]

    # Finalize the assembly of the block matrix
    result.assemble()
    return result

# Coarsest mesh eigenvalues

In [92]:
# Read the matrix data
# i = 0
A_data = np.loadtxt("../../build/A0.dat")
B_data = np.loadtxt("../../build/B0.dat")

# Create the PETSc matrices A and B
A = create_petsc_matrix(A_data)
B = create_petsc_matrix(B_data)

# OPTIONAL FOR 3FIELD
# A = preprocess_A_3field(A_data, 1854, 2808, 1296)
# B = preprocess_B_3field(B_data, 1854, 2808, 1296)

# Compute evals
# evals = eigenvalues(A, B, 0.5, 343+200) # dim Lagr = 343
# evals = eigenvalues(A, B, 0.01, 100)
# evals = eigenvalues(A, B, 1.1, 11, True)

evals = eigenvalues(A, B, 3.2, 41)

# evals = evals[..., None]
np.savetxt(sys.stdout, evals[6:20], fmt="%.1f")

Number of iterations: 5
Solution method: krylovschur
Number of requested eigenvalues: 41
Stopping condition: tol=1e-16, maxit=145
Number of converged eigenpairs: 41
Exact    = [2. 2. 2. 3. 3. 5. 5. 5. 5. 5. 5.]
Nédélec  = [-2.26000e-02 -1.79600e-02 -1.51400e-02 -1.33900e-02 -8.67000e-03
 -8.58000e-03 -7.84000e-03 -3.33000e-03 -4.50000e-04  2.90000e-04
  1.15700e-02  1.57400e-02  1.71900e-02  1.76800e-02  1.91400e-02
  2.46700e-02  3.03200e-02  3.74300e-02  4.72100e-02  7.13600e-02
  7.92900e-02  1.02000e-01  1.20080e-01  1.21720e-01  1.93365e+00
  1.97670e+00  2.01451e+00  2.99473e+00  3.04135e+00  4.53125e+00
  4.62748e+00  4.74872e+00  4.91161e+00  4.98472e+00  5.06898e+00
  5.74538e+00  5.82865e+00  5.89176e+00  5.95026e+00  6.12526e+00
  6.21003e+00]
-0.0
-0.0
-0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.1


# Finest mesh eigenvalues

In [99]:
# i = 1
A_data = np.loadtxt("../../build/A1.dat")
B_data = np.loadtxt("../../build/B1.dat")

# Create the PETSc matrices A and B
A = create_petsc_matrix(A_data)
B = create_petsc_matrix(B_data)

# evals = eigenvalues(A, B, 0.5, 1728+20) # dim Lagr = 2197
# evals = eigenvalues(A, B, 0.8, 100)
# evals = eigenvalues(A, B, 1.1, 11, True)

evals = eigenvalues(A, B, 3.2, 41)

np.savetxt(sys.stdout, evals, fmt="%.1f")
# np.savetxt(sys.stdout, evals[1:30], fmt="%.1f")

Number of iterations: 74
Solution method: krylovschur
Number of requested eigenvalues: 41
Stopping condition: tol=1e-16, maxit=1107
Number of converged eigenpairs: 41
Exact    = [2. 2. 2. 3. 3. 5. 5. 5. 5. 5. 5.]
Nédélec  = [1.66000e-03 1.66000e-03 1.70000e-03 1.74000e-03 1.78000e-03 1.78000e-03
 1.78000e-03 1.89000e-03 1.90000e-03 1.92000e-03 2.01000e-03 2.04000e-03
 2.05000e-03 2.09000e-03 2.40000e-03 2.44000e-03 2.51000e-03 2.77000e-03
 2.85000e-03 3.18000e-03 3.80000e-03 4.47000e-03 5.04000e-03 5.66000e-03
 1.97919e+00 1.99065e+00 2.00411e+00 2.99704e+00 3.01086e+00 4.85003e+00
 4.85596e+00 4.92379e+00 4.97249e+00 4.99048e+00 5.01561e+00 5.92063e+00
 5.93711e+00 5.94859e+00 5.98010e+00 6.03819e+00 6.05239e+00]
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
2.0
2.0
2.0
3.0
3.0
4.9
4.9
4.9
5.0
5.0
5.0
5.9
5.9
5.9
6.0
6.0
6.1


# Printing eigenvalue results

In [101]:
# np.savetxt(sys.stdout, evals[0:20], fmt="%.2f")

np.savetxt(sys.stdout, evals[20:-1], fmt="%.2f")


0.00
0.00
0.01
0.01
1.98
1.99
2.00
3.00
3.01
4.85
4.86
4.92
4.97
4.99
5.02
5.92
5.94
5.95
5.98
6.04


# Debugging matrices

In [13]:
Ad = A.convert('dense').getDenseArray()
Bd = B.convert('dense').getDenseArray()

size = Bd.shape
# Bd[size[0]-2,size[1]-2]
# np.diag(Bd)[2808:size[0]]

# Ad[1968-1, 1968-1]
# Ad.diagonal()[1968-21 : 1968+21]
print(Ad.shape)
Ad[5520:6000,5520:5525]

(5522, 5522)


array([[0., 0.],
       [0., 0.]])